<a href="https://colab.research.google.com/github/sherjahong1r/Machine-Learning-Lessons/blob/main/14_Filmlar_sharhi_(pos_neg)_Amaliy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Modelni moslashtirish**

## **Avval shug'ullantirilgan modelni o'z ma'lumotimizga moslashtirish**

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
!pip install -U datasets fsspec

In [2]:
!pip install -U transformers

  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)
Using cached transformers-5.3.0-py3-none-any.whl (10.7 MB)
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


### Ushbu notebookdagi model matnni sentiment tahlil qilishga qaratilgan. Ya'ni, u berilgan matnning ijobiy yoki salbiy kayfiyatda ekanligini aniqlash uchun tayyorlangan.


### **Qisqacha aytganda:**

### Model turi: Bu DistilBERT nomli oldindan o'rgatilgan til modeli bo'lib, u ketma-ketliklarni tasniflash (sequence classification) uchun moslashtirilgan.

### Ma'lumotlar to'plami: Model SST-2 ma'lumotlar to'plamida (The Stanford Sentiment Treebank) qayta o'qitilgan (fine-tuned). Bu to'plam ingliz tilidagi filmlar sharhlarini o'z ichiga oladi va ularni ijobiy yoki salbiy deb belgilaydi.

### Nima bashorat qiladi: Model yangi matnlarni qabul qilib, ularning ijobiy (Ijobiy) yoki salbiy (Salbiy) ekanligini bashorat qiladi. Masalan, "this movie is a masterpiece" jumlasi "Ijobiy" deb, "a complete waste of time" jumlasi esa "Salbiy" deb baholandi.



# **1. Ma’lumotlar to‘plamini yuklash va tayyorlash**





### datasets kutubxonasidan foydalanib, sst2 ma’lumotlar to‘plamini yuklang.




### Modelni tezroq o‘rgatish uchun to‘plamdan kichikroq qismlarni ajratib oling: train uchun 2000 ta va test uchun 400 ta misol.



### seed=42 parametrini ishlating.



In [3]:
from datasets import load_dataset

dataset = load_dataset("sst2")
small_train = dataset["train"].shuffle(seed=42).select(range(2000))
small_test = dataset['test'].shuffle(seed=42).select(range(400))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

# **2. Matnni tokenizatsiya qilish**





### distilbert-base-uncased tokenizer’ini yuklang.



### sst2 to‘plamidagi matnlar qisqaroq bo‘lgani uchun max_length=128 qilib, preprocess_function yarating.



### Bu funksiyani small_train va small_test to‘plamlariga map metodi orqali qo‘llang.

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def preprocess_function(examples):
    tokenized_inputs = tokenizer(examples["sentence"], truncation=True, padding='max_length', max_length=128)
    tokenized_inputs["labels"] = examples["label"]
    return tokenized_inputs

tokenized_train = small_train.map(preprocess_function, batched=True)
tokenized_test = small_test.map(preprocess_function, batched=True)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

In [5]:
print(small_train.column_names)
print(small_test.column_names)

['idx', 'sentence', 'label']
['idx', 'sentence', 'label']


# **3. Modelni shug‘ullantirish**





### distilbert-base-uncased asosida ketma-ketlikni tasniflash uchun model (AutoModelForSequenceClassification) yuklang.



### TrainingArguments sozlamalarini yangi qiymatlar bilan yarating: num_train_epochs=2, logging_steps=50.



### Trainer obyektini yarating va train() metodi bilan modelni shug‘ullantiring.

In [6]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification
import os
import torch

model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

# Modelni aniq CPU ga o'tkazish (Agar yuqoridagi muhit o'zgaruvchisi yetarli bo'lmasa)
# model.to('cpu') kodini faqat CUDA mavjud bo'lganda ishlatish maqsadga muvofiq.
# Agar CUDA_VISIBLE_DEVICES="" o'rnatilgan bo'lsa, PyTorch CUDA ni ko'rmaydi.
# Lekin agar model hali ham GPU da bo'lsa, uni CPU ga o'tkazish kerak.
if torch.cuda.is_available() and model.device.type == 'cuda':
    model.to('cpu')

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=50,
    report_to='none'
    )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test
)

trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
50,0.525409
100,0.337083
150,0.281245
200,0.194621
250,0.174912


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.3026542091369629, metrics={'train_runtime': 1503.1552, 'train_samples_per_second': 2.661, 'train_steps_per_second': 0.166, 'total_flos': 132467398656000.0, 'train_loss': 0.3026542091369629, 'epoch': 2.0})

# **4. Modelni baholash va sinovdan o‘tkazish**





### trainer.evaluate() metodini chaqirib, modelning eval_loss ko‘rsatkichini tahlil qiling.



### yangi matnlar ("this movie is a masterpiece", "a complete waste of time") yaratib, modelni shu matnlarda sinab ko‘ring.



### Natijalarni (matn, belgi, ehtimollik) ekranga chiqaring.

In [8]:
# trainer.evaluate()


In [12]:
texts = [
    "this movie is a masterpiece", "a complete waste of time"
]

In [13]:
import torch
import torch.nn.functional as F

In [14]:
import torch
import torch.nn.functional as F

# Inputs ni to'g'ridan-to'g'ri CPU ga yuborish
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=256).to('cpu')

# Modelni ham CPU ga o'tkazilganligiga ishonch hosil qilish
if torch.cuda.is_available() and model.device.type == 'cuda':
    model.to('cpu')

with torch.no_grad():
    outputs = model(**inputs)
    probs = F.softmax(outputs.logits, dim=1)
    predictions = torch.argmax(probs, dim=1)

label_map = {0: "negative", 1: 'possitive'}

for text, pred, prob in zip(texts, predictions, probs):
    print(text, label_map[pred.item()], prob[pred.item()].item())

this movie is a masterpiece possitive 0.9953193068504333
a complete waste of time negative 0.9785634279251099
